# 7.18 後訓練有哪些不同教法？


你想教模型回答「只回數字：1+2=?」，可以直接示範`3`，也可以拿`3`和`答案是3喔！`請人比較，或讓模型先作答，再檢查是否正確且符合格式。三種方式都能提供教學訊號，但給模型的材料不同。先看材料，之後再記縮寫，會比較容易理解每個方法在解決什麼問題。

前置是[7.1的對話資料](https://birdhackor.github.io/tiny-perceptron-vlm/7.1.html)與[5.8的實際回答檢查](https://birdhackor.github.io/tiny-perceptron-vlm/5.8.html)：代價下降後，還要另外查看模型是否完成題目。**示範**給一份期望模型學會的回答；**偏好**給同題兩篇回答以及哪篇比較合適；**回饋**給模型實際作答之後的評分。評分叫reward，中文常譯獎勵；它是一個用來指引更新方向的數字，不是模型的情緒，也不自動代表真實品質。下面用三張資料卡把它們分開。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
demonstration = {"question": "只回數字：1+2=?", "answer": "3"}
preference = {"question": demonstration["question"], "chosen": "3", "rejected": "答案是3喔！"}
feedback = {"question": demonstration["question"], "sampled_answer": "4", "reward": 0}
print("示範給什麼", demonstration["answer"])
print("偏好比較什麼", preference["chosen"], "勝過", preference["rejected"])
print("作答後得到什麼", feedback["sampled_answer"], feedback["reward"])

輸入是我們人工寫的字典；`chosen`表示被選為較佳、`rejected`表示較差，`sampled_answer`記錄作答，這裡只是用`4`示意，沒有真的抽樣模型。輸出依序是示範`3`、偏好`3`勝過附加句、錯答`4`得到0。偏好兩篇都算對，但後者違反只回數字；回饋則採這個例子的正確性規則。換任務時，不能沿用數字0就假裝已定義完整標準。

用理想回答繼續訓練就是**SFT，監督式微調**；我們已在本章追過它的答案位置與loss。用偏好對直接調整回答傾向的一種方法叫**DPO，直接偏好最佳化**，見[13.3–13.6](https://birdhackor.github.io/tiny-perceptron-vlm/13.3.html)。它不需要另外訓練一位評分員，仍需要可靠的比較資料與固定參考。這裡的**固定參考**是開始這段偏好訓練時保留、不再更新的模型副本，用來比較回答傾向相對起點改了多少。

也可以先用偏好對教一個**獎勵模型**，讓它估計哪篇回答更合適，再讓回答模型取樣、得到評分、更新選擇。**RLHF**的全名是reinforcement learning from human feedback，從人類回饋做強化學習；典型流程可見[InstructGPT的3.1節](https://arxiv.org/pdf/2203.02155v1)。其中更新策略的一種方法叫**PPO，近端策略最佳化**。RLHF說明回饋的來源與流程，PPO說明怎樣更新，兩者不是同一個名詞。若回饋來自程式驗算，則是[C.6的可驗證回饋](https://birdhackor.github.io/tiny-perceptron-vlm/C.6.html)，不能因為用了PPO就改稱人類回饋。

圖中的三張卡分別送往示範學習、直接偏好學習與評分後更新；中間的偏好卡也能先用來訓練獎勵模型。這些是可選路線，沒有要求每個模型依序把SFT、PPO、DPO都跑一次。第13章先用兩篇回答建立偏好直覺，再逐步展開這些名字。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/posttrain_signals.svg")))

後訓練能改變的不只語氣。若示範包含解題步驟，或回饋能辨認答案正誤，它也可能改善模型做某些任務的能力；成功與否仍要用新題檢查。另一個實際配方是[DeepSeek-R1論文2.2與2.3節](https://arxiv.org/pdf/2501.12948v1)：一條路線從已完成預訓練的**底座模型**直接用作答回饋做**強化學習，英文縮寫RL**；另一條先用少量理想解題示範，教出清楚的起始回答方式，再接後續訓練。這批起步時加入的示範稱為**冷啟動資料**。兩條路線說明SFT不是所有RL配方不可省略的前一步，不代表我們的微型模型能重現論文能力。

練習把問題改成「用一句話解釋1+2」，先寫出新的示範，並重新判斷原偏好是否還合理。光交換`chosen`和`rejected`不一定足夠：原附加句沒有真的解釋加法。最後說明若用真人比較、程式驗算或另一個模型評分，回饋的來源各是哪一種；方法縮寫不應遮住這個差別。
